In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("EmployeeAttendanceAnalysis") \
    .getOrCreate()

In [2]:
from pyspark.sql.functions import (
    col,
    avg,
    when
)

In [3]:
attendance_data = [
    (101, "Rahul", "Sales", "09:05", 8.5, 95, "Present"),
    (102, "Priya", "HR", "09:45", 7.2, 82, "Present"),
    (103, "Amit", "IT", "08:55", 9.0, 98, "Present"),
    (104, "Sneha", "Finance", "10:10", 6.5, 70, "Present"),
    (105, "Kiran", "Sales", None, 0.0, 0, "Absent"),
    (106, "Divya", "HR", "09:20", 8.0, 88, "Present"),
    (107, "Naveen", "IT", None, 0.0, 0, "Absent"),
    (108, "Megha", "Finance", "08:50", 9.1, 96, "Present")
]

columns = [
    "employee_id",
    "employee_name",
    "department",
    "login_time",
    "work_hours",
    "productivity",
    "status"
]

attendance_df = spark.createDataFrame(attendance_data, columns)

In [4]:
attendance_df.show(truncate=False)

+-----------+-------------+----------+----------+----------+------------+-------+
|employee_id|employee_name|department|login_time|work_hours|productivity|status |
+-----------+-------------+----------+----------+----------+------------+-------+
|101        |Rahul        |Sales     |09:05     |8.5       |95          |Present|
|102        |Priya        |HR        |09:45     |7.2       |82          |Present|
|103        |Amit         |IT        |08:55     |9.0       |98          |Present|
|104        |Sneha        |Finance   |10:10     |6.5       |70          |Present|
|105        |Kiran        |Sales     |NULL      |0.0       |0           |Absent |
|106        |Divya        |HR        |09:20     |8.0       |88          |Present|
|107        |Naveen       |IT        |NULL      |0.0       |0           |Absent |
|108        |Megha        |Finance   |08:50     |9.1       |96          |Present|
+-----------+-------------+----------+----------+----------+------------+-------+



In [5]:
late_logins = attendance_df.filter(
    col("login_time") > "09:15"
)

print("Late Login Employees")
late_logins.show()

Late Login Employees
+-----------+-------------+----------+----------+----------+------------+-------+
|employee_id|employee_name|department|login_time|work_hours|productivity| status|
+-----------+-------------+----------+----------+----------+------------+-------+
|        102|        Priya|        HR|     09:45|       7.2|          82|Present|
|        104|        Sneha|   Finance|     10:10|       6.5|          70|Present|
|        106|        Divya|        HR|     09:20|       8.0|          88|Present|
+-----------+-------------+----------+----------+----------+------------+-------+



In [6]:
absent_employees = attendance_df.filter(
    col("status") == "Absent"
)

print("Absent Employees")
absent_employees.show()

Absent Employees
+-----------+-------------+----------+----------+----------+------------+------+
|employee_id|employee_name|department|login_time|work_hours|productivity|status|
+-----------+-------------+----------+----------+----------+------------+------+
|        105|        Kiran|     Sales|      NULL|       0.0|           0|Absent|
|        107|       Naveen|        IT|      NULL|       0.0|           0|Absent|
+-----------+-------------+----------+----------+----------+------------+------+



In [7]:
attendance_issues = attendance_df.filter(
    (col("login_time") > "09:15") |
    (col("status") == "Absent")
)

print("Attendance Issues")
attendance_issues.show()

Attendance Issues
+-----------+-------------+----------+----------+----------+------------+-------+
|employee_id|employee_name|department|login_time|work_hours|productivity| status|
+-----------+-------------+----------+----------+----------+------------+-------+
|        102|        Priya|        HR|     09:45|       7.2|          82|Present|
|        104|        Sneha|   Finance|     10:10|       6.5|          70|Present|
|        105|        Kiran|     Sales|      NULL|       0.0|           0| Absent|
|        106|        Divya|        HR|     09:20|       8.0|          88|Present|
|        107|       Naveen|        IT|      NULL|       0.0|           0| Absent|
+-----------+-------------+----------+----------+----------+------------+-------+



In [8]:
department_summary = attendance_df.groupBy("department").agg(
    avg("work_hours").alias("avg_work_hours"),
    avg("productivity").alias("avg_productivity")
)

department_summary.show()

+----------+--------------+----------------+
|department|avg_work_hours|avg_productivity|
+----------+--------------+----------------+
|     Sales|          4.25|            47.5|
|        HR|           7.6|            85.0|
|   Finance|           7.8|            83.0|
|        IT|           4.5|            49.0|
+----------+--------------+----------------+



In [9]:
from pyspark.sql.functions import round

department_summary = department_summary.select(
    "department",
    round(col("avg_work_hours"), 2).alias("average_work_hours"),
    round(col("avg_productivity"), 2).alias("average_productivity")
)

department_summary.show()

+----------+------------------+--------------------+
|department|average_work_hours|average_productivity|
+----------+------------------+--------------------+
|     Sales|              4.25|                47.5|
|        HR|               7.6|                85.0|
|   Finance|               7.8|                83.0|
|        IT|               4.5|                49.0|
+----------+------------------+--------------------+



In [10]:
attendance_issues.coalesce(1) \
    .write.mode("overwrite") \
    .option("header", True) \
    .csv("/content/attendance_issues")

In [11]:
department_summary.coalesce(1) \
    .write.mode("overwrite") \
    .option("header", True) \
    .csv("/content/department_summary")

In [12]:
print("===== Attendance Issues =====")
attendance_issues.show(truncate=False)

print("===== Department Summary =====")
department_summary.show(truncate=False)

===== Attendance Issues =====
+-----------+-------------+----------+----------+----------+------------+-------+
|employee_id|employee_name|department|login_time|work_hours|productivity|status |
+-----------+-------------+----------+----------+----------+------------+-------+
|102        |Priya        |HR        |09:45     |7.2       |82          |Present|
|104        |Sneha        |Finance   |10:10     |6.5       |70          |Present|
|105        |Kiran        |Sales     |NULL      |0.0       |0           |Absent |
|106        |Divya        |HR        |09:20     |8.0       |88          |Present|
|107        |Naveen       |IT        |NULL      |0.0       |0           |Absent |
+-----------+-------------+----------+----------+----------+------------+-------+

===== Department Summary =====
+----------+------------------+--------------------+
|department|average_work_hours|average_productivity|
+----------+------------------+--------------------+
|Sales     |4.25              |47.5      